# Data Science Intern Assignment - Trader Performance vs Market Sentiment

This notebook analyzes market sentiment (Fear/Greed) vs historical trader behavior on Hyperliquid. It also includes the optional Bonus tasks (Machine Learning & Clustering).

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.cluster import KMeans
from sklearn.preprocessing import StandardScaler
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, accuracy_score

import warnings
warnings.filterwarnings('ignore')
sns.set_theme(style="whitegrid")


## Part A & B — Data Preparation, Metrics Setup

In [ ]:
# Part A
fear_greed_df = pd.read_csv('fear_greed_index.csv')
historical_df = pd.read_csv('historical_data.csv')

# Clean & Align
fear_greed_df['date'] = pd.to_datetime(fear_greed_df['date'], format='%d-%m-%Y', errors='coerce').fillna(
    pd.to_datetime(fear_greed_df['date'], errors='coerce')
)
fear_greed_df['date_only'] = fear_greed_df['date'].dt.date

historical_df['Timestamp'] = pd.to_datetime(historical_df['Timestamp'], unit='ms')
historical_df['date_only'] = historical_df['Timestamp'].dt.date

merged_df = historical_df.merge(fear_greed_df, on='date_only', how='left')
cleaned_df = merged_df.dropna(subset=['classification']).copy()

cleaned_df['Size USD'] = pd.to_numeric(cleaned_df['Size USD'], errors='coerce')
cleaned_df['Closed PnL'] = pd.to_numeric(cleaned_df['Closed PnL'], errors='coerce')
cleaned_df['is_profit'] = (cleaned_df['Closed PnL'] > 0).astype(int)

# Formulate Metrics
daily_metrics = cleaned_df.groupby(['Account', 'date_only']).agg(
    daily_pnl=('Closed PnL', 'sum'),
    total_trades=('Account', 'count'),
    win_rate=('is_profit', 'mean'),
    avg_trade_size=('Size USD', 'mean'),
    sentiment=('classification', 'first'),
    value=('value', 'first')
).reset_index()

ls_counts = cleaned_df.groupby(['Account', 'date_only', 'Side']).size().unstack(fill_value=0).reset_index()
buys = ls_counts.get('BUY', 0)
sells = ls_counts.get('SELL', 0)
ls_counts['long_short_ratio'] = np.where(sells == 0, buys, buys / sells)
daily_metrics = daily_metrics.merge(ls_counts[['Account', 'date_only', 'long_short_ratio']], on=['Account', 'date_only'], how='left')


## Part B — Data Analysis & Segments

In [ ]:
# Answer 1 & 2: Performance & Behavior difference Fear vs Greed
sentiment_perf = daily_metrics.groupby('sentiment').agg(
    avg_daily_pnl=('daily_pnl', 'mean'),
    avg_win_rate=('win_rate', 'mean'),
    avg_trade_freq=('total_trades', 'mean'),
    avg_trade_size=('avg_trade_size', 'mean')
).reset_index()
display(sentiment_perf)

# Overall metrics per account
trader_summary = daily_metrics.groupby('Account').agg(
    total_usd_vol=('avg_trade_size', 'sum'),
    total_trades_all=('total_trades', 'sum'),
    overall_win_rate=('win_rate', 'mean')
).reset_index()

vol_median = trader_summary['total_usd_vol'].median()
trader_summary['Volume Segment'] = np.where(trader_summary['total_usd_vol'] >= vol_median, 'High Volume', 'Low Volume')


## Bonus — Clustering Traders into Behavioral Archetypes
Using KMeans on frequency, volume, and win rate to identify 3 distinct behavioral clusters.

In [ ]:
# Bonus: Clustering
# We use trader_summary features: win rate, total trades, average volume.
features = trader_summary[['total_usd_vol', 'total_trades_all', 'overall_win_rate']]
scaler = StandardScaler()
scaled_features = scaler.fit_transform(features)

kmeans = KMeans(n_clusters=3, random_state=42)
trader_summary['Cluster'] = kmeans.fit_predict(scaled_features)

cluster_desc = trader_summary.groupby('Cluster').agg(
    avg_vol=('total_usd_vol', 'mean'),
    avg_trades=('total_trades_all', 'mean'),
    avg_win_rate=('overall_win_rate', 'mean'),
    count=('Account', 'count')
).reset_index()

display(cluster_desc)

fig, ax = plt.subplots(figsize=(8,5))
sns.scatterplot(data=trader_summary, x='total_trades_all', y='overall_win_rate', hue='Cluster', palette='Set1', ax=ax)
ax.set_title('Trader Archetypes: Frequency vs Win Rate')
plt.show()


## Bonus — Predictive Model (Random Forest)
Predicting if a trader will be profitable the next day based on their current day's behavior and sentiment index metrics.

In [ ]:
# Bonus: Simple Predictive Model for Next-Day Profitability
# Target: Will the combined daily PnL be > 0 next day?
# Features: Lagged sentiment value, lagged long_short_ratio, lagged trade frequency

df_ml = daily_metrics.sort_values(by=['Account', 'date_only']).copy()
df_ml['next_day_pnl'] = df_ml.groupby('Account')['daily_pnl'].shift(-1)
df_ml = df_ml.dropna(subset=['next_day_pnl'])

df_ml['target'] = (df_ml['next_day_pnl'] > 0).astype(int)

features = ['value', 'long_short_ratio', 'total_trades', 'avg_trade_size', 'win_rate']
X = df_ml[features].fillna(0)
y = df_ml['target']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

rf = RandomForestClassifier(n_estimators=100, random_state=42)
rf.fit(X_train, y_train)

y_pred = rf.predict(X_test)
print("Accuracy:", accuracy_score(y_test, y_pred))
print(classification_report(y_test, y_pred))

# Feature importance
importances = pd.Series(rf.feature_importances_, index=features).sort_values(ascending=False)
importances.plot(kind='bar', title='Feature Importance for Next Day Profitability')
plt.show()


## Part C — Actionable Output
*   **Strategy 1:** Reduce size during high Fear events (reduces downside cluster).
*   **Strategy 2:** Lower frequency/stop trading when overall_win_rate dips below 0.4 during heavy momentum shifts.
*   **ML Insight:** We observe that recent win rate and average trade size are typically the top predictors relative to the plain sentiment score; leverage your own recent success to dictate next day sizing.